# Ordenación por selección (selection sort)

En cada paso busca el mínimo de la parte sin ordenar y lo coloca al principio de esta. Hace $O(n^2)$ comparaciones pero solo $O(n)$ intercambios.

## 1. Selección

Ordenar una lista de menor a mayor por selección, modificándola en el sitio.

In [1]:
import random

def selection_sort(l):
    for i in range(len(l)):
        minimo = min(range(i, len(l)), key=l.__getitem__)
        l[i], l[minimo] = l[minimo], l[i]
    return l

for _ in range(200):
    datos = random.choices(range(100), k=random.randint(0, 30))
    assert selection_sort(datos[:]) == sorted(datos)
print(selection_sort([64, 25, 12, 22, 11]))

[11, 12, 22, 25, 64]


## 2. Las k menores

Usar la idea de la selección para obtener solo los $k$ elementos menores, ordenados, sin ordenar toda la lista.

In [2]:
def k_menores(lista, k):
    l = lista[:]
    for i in range(min(k, len(l))):
        minimo = min(range(i, len(l)), key=l.__getitem__)
        l[i], l[minimo] = l[minimo], l[i]
    return l[:k]

print(k_menores([64, 25, 12, 22, 11, 90, 3], 3))

[3, 11, 12]


## 3. Estabilidad

La selección no es estable: al intercambiar el mínimo con el primero de la parte sin ordenar, ese primero puede saltar por encima de otro igual. Comprobarlo ordenando alumnos por nota y hacerla estable desplazando el mínimo a su sitio (`pop` + `insert`) en lugar de intercambiarlo, a cambio de mover $O(n^2)$ elementos.

In [3]:
def nota(alumno):
    return alumno[1]

def primero(registro):
    return registro[0]

def selection_sort_clave(lista, clave):
    l = lista[:]
    for i in range(len(l)):
        minimo = min(range(i, len(l)), key=lambda j: clave(l[j]))
        l[i], l[minimo] = l[minimo], l[i]
    return l

def selection_sort_estable(lista, clave):
    l = lista[:]
    for i in range(len(l)):
        minimo = min(range(i, len(l)), key=lambda j: clave(l[j]))
        # desplaza en vez de intercambiar: el resto conserva su orden
        l.insert(i, l.pop(minimo))
    return l

alumnos = [("Ana", 7), ("Eva", 7), ("Luis", 5), ("Pau", 10)]
print("Intercambiando:", selection_sort_clave(alumnos, nota))
print("Desplazando:   ", selection_sort_estable(alumnos, nota))
assert selection_sort_clave(alumnos, nota) != sorted(alumnos, key=nota)

random.seed(3)
for _ in range(300):
    datos = [(random.randint(0, 5), i) for i in range(random.randint(0, 20))]
    assert selection_sort_estable(datos, primero) == sorted(datos, key=primero)

Intercambiando: [('Luis', 5), ('Eva', 7), ('Ana', 7), ('Pau', 10)]
Desplazando:    [('Luis', 5), ('Ana', 7), ('Eva', 7), ('Pau', 10)]


## 4. Comparaciones e intercambios

Contar comparaciones e intercambios. La selección hace siempre $\frac{n(n-1)}{2}$ comparaciones (su mejor y su peor caso coinciden), pero como mucho $n - 1$ intercambios, útil cuando escribir es caro. Compararlo con las inversiones de la lista, que son los intercambios que haría la burbuja.

In [4]:
def selection_sort_contando(lista):
    l, comparaciones, intercambios = lista[:], 0, 0
    for i in range(len(l)):
        minimo = i
        for j in range(i + 1, len(l)):
            comparaciones += 1
            if l[j] < l[minimo]:
                minimo = j
        if minimo != i:
            l[i], l[minimo] = l[minimo], l[i]
            intercambios += 1
    return l, comparaciones, intercambios

def inversiones(lista):
    return sum(lista[i] > lista[j] for i in range(len(lista)) for j in range(i + 1, len(lista)))

random.seed(4)
n = 100
casos = {"ordenada": list(range(n)), "inversa": list(range(n, 0, -1)), "aleatoria": random.sample(range(n), n)}
print(f"{'caso':>10} {'comparaciones':>14} {'intercambios':>13} {'inversiones':>12}")
for nombre, datos in casos.items():
    l, comparaciones, intercambios = selection_sort_contando(datos)
    assert l == sorted(datos) and comparaciones == n * (n - 1) // 2 and intercambios <= n - 1
    print(f"{nombre:>10} {comparaciones:>14} {intercambios:>13} {inversiones(datos):>12}")
assert selection_sort_contando(casos["ordenada"])[2] == 0
assert selection_sort_contando(casos["inversa"])[2] == n // 2

      caso  comparaciones  intercambios  inversiones
  ordenada           4950             0            0
   inversa           4950            50         4950
 aleatoria           4950            92         2058
